# Benchmarking Early Time Series Classification
## Under the Earliness–Reliability–Stability Trilemma

**Author:** Moncef Bouhabel  
**Context:** Preparatory work for CIFRE PhD — Orange Innovation / EURECOM (ref. 2026-51517)  
**Supervisors (industrial):** Dr. Lionel Tailhardat, Dr. Raphaël Troncy  

---

## 1. Problem Statement

In network anomaly detection and cybersecurity incident triage, classification systems face a fundamental tension:

> *How early can we commit to a diagnosis — without sacrificing reliability?*

This notebook benchmarks two published early classification methods — **TEASER** and **ECONOMY-K** — on standard UCR time series datasets, and jointly evaluates them on three dimensions:

| Dimension | Definition | Relevance to NetOps |
|---|---|---|
| **Earliness** | Fraction of series unseen at decision | Faster incident escalation |
| **Reliability** | Classification accuracy at trigger time | Fewer false positives |
| **Stability** | Decision consistency across runs | Trusted, auditable system |

The three objectives are **mutually conflicting** — optimising one typically degrades the others. This is the *trilemma*.

Understanding the Pareto frontier between these objectives is a prerequisite for designing **adaptive multi-model orchestration systems** — the core scientific objective of the target CIFRE thesis.

---
## 2. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

from datasets import load_dataset, UCR_DATASETS
from metrics import evaluate, harmonic_mean_score

sns.set_theme(style='whitegrid', font_scale=1.2)
print('Setup complete.')

---
## 3. Datasets

We select 5 datasets from the UCR Time Series Archive (Dau et al., 2018), chosen for their diversity of domains and relevance to infrastructure monitoring:

In [ ]:
DATASETS = ['ItalyPowerDemand', 'ECG200', 'SonyAIBORobotSurface1', 'ECG5000', 'Wafer']

rows = []
for name in DATASETS:
    X_train, y_train, X_test, y_test = load_dataset(name)
    rows.append({
        'Dataset': name,
        'Domain': UCR_DATASETS[name]['domain'],
        'Train': len(X_train),
        'Test': len(X_test),
        'T (length)': X_train.shape[1],
        'Classes': len(np.unique(y_train))
    })

pd.DataFrame(rows).set_index('Dataset')

**Why these datasets?**
- **ItalyPowerDemand** (T=24): short series, 2 classes — baseline for fast classification
- **ECG200 / ECG5000** (T=96, 140): physiological signals, safety-critical — analogous to network health signals
- **SonyAIBORobotSurface1** (T=70): sensor data under variable conditions — analogous to noisy telemetry
- **Wafer** (T=152): industrial fault detection with class imbalance — directly analogous to anomaly detection in production

---
## 4. Methods

### TEASER (Schäfer & Leser, 2020)
Two-tier architecture: a WEASEL-based time series classifier generates probabilities at each timestep, and an OCSVM (One-Class SVM) acts as a stopping rule — it accepts a prediction when the probability distribution is sufficiently "normal" (i.e., confident). If the OCSVM rejects, the classifier waits for the next timestep.

**Key property:** non-myopic stopping — the decision to classify now accounts for future information.

### ECONOMY-K (Achenchabe et al., 2021)
Cost-sensitive stopping rule: at each timestep *t*, classify using 1-NN and trigger when the confidence exceeds a threshold *τ*. The threshold encodes the relative cost of misclassification vs. late classification.

**Key property:** explicit cost model — the threshold *τ* directly controls the accuracy–earliness trade-off.

> **Note:** Alexis Bondu, a co-author of the ECONOMY-K paper, is a researcher at Orange Innovation. This benchmark directly engages with his published work.

---
## 5. Results

### 5.1 Raw metrics

In [ ]:
df = pd.read_csv('../results/benchmark_results_full.csv')

pivot = df.pivot_table(
    index='dataset', columns='method',
    values=['accuracy', 'earliness', 'hm_score']
).round(3)

pivot

### 5.2 Summary statistics

In [ ]:
summary = df.groupby('method')[['accuracy', 'earliness', 'hm_score']].agg(['mean', 'std']).round(3)
print('Mean ± Std across 5 datasets:\n')
summary

### 5.3 Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
figs = [
    ('fig1_pareto_scatter.png',  'Fig 1 — Accuracy vs Earliness (Pareto)'),
    ('fig2_hm_score_bars.png',   'Fig 2 — HM Score by Dataset'),
    ('fig3_radar_profile.png',   'Fig 3 — Global Method Profile'),
    ('fig4_wins_heatmap.png',    'Fig 4 — Head-to-Head Heatmap'),
]

for ax, (fname, title) in zip(axes.flat, figs):
    img = mpimg.imread(f'../results/figures/{fname}')
    ax.imshow(img)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')

fig.suptitle('Trilemma Benchmark — TEASER vs ECONOMY-K', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Threshold Sensitivity Analysis (ECONOMY-K)

The confidence threshold *τ* is ECONOMY-K's main hyperparameter. Understanding how it controls the trilemma is essential for designing adaptive systems.

**Research question:** Is there a single optimal threshold, or does the optimal *τ* vary per dataset?

In [ ]:
try:
    sens = pd.read_csv('../results/sensitivity_results.csv')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    figs_sens = [
        ('fig5_threshold_hm_score.png',       'Fig 5 — HM Score vs Threshold'),
        ('fig6_pareto_threshold_frontier.png', 'Fig 6 — Pareto Frontier vs Threshold'),
    ]
    for ax, (fname, title) in zip(axes, figs_sens):
        img = mpimg.imread(f'../results/figures/{fname}')
        ax.imshow(img)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    print('\nOptimal threshold per dataset (max HM Score):\n')
    optimal = sens.loc[sens.groupby('dataset')['hm_score'].idxmax()]
    print(optimal[['dataset','threshold','accuracy','earliness','hm_score']].to_string(index=False))

    mean_hm = sens.groupby('threshold')['hm_score'].mean()
    best = mean_hm.idxmax()
    print(f'\nGlobal optimal threshold: {best:.2f} (mean HM = {mean_hm[best]:.3f})')
    print('\nNote: HM plateaus for τ < 0.55 — 1-NN confidence saturates at low thresholds.')
    print('Effective decision zone: τ ∈ [0.55, 0.70]')

except FileNotFoundError:
    print('Run src/sensitivity_analysis.py first.')

---
## 7. Key Findings

### Finding 1 — No universally dominant method
ECONOMY-K wins on 4/5 datasets (HM score), but TEASER achieves higher earliness on ECG200 (0.917 vs 0.899). The best method depends on dataset characteristics — confirming that **static method selection is suboptimal**.

### Finding 2 — Similar earliness, different accuracy
Both methods achieve ~0.87 mean earliness, but ECONOMY-K maintains significantly higher accuracy (0.807 vs 0.726). TEASER's non-myopic stopping rule does not consistently translate into better accuracy.

### Finding 3 — Confidence metric saturation
The 1-NN confidence metric saturates below τ = 0.55 across all datasets. The effective decision zone is τ ∈ [0.55, 0.70], motivating research on richer confidence estimators.

### Finding 4 — Stability is deterministic
Both methods produce identical trigger times across 3 random seeds (stability = 0). ECONOMY-K is deterministic by design; TEASER's OCSVM is robust to initialisation on these datasets.

---
## 8. Heterogeneous Orchestration — Experimental Journey

Having established that no single method dominates, we prototype orchestration layers combining agents. This section documents two experiments — including one negative result. Both are scientifically valuable.

### Experiment 1 — TEASER + ECONOMY-K (Negative Result)

Three cooperation schemes: **OR** (either triggers), **AND** (both agree), **WEIGHTED** (vote by accuracy).

**Result:** TEASER systematically returns -1 (abstain) on partial series in small-sample settings. Its OCSVM stopping rule, calibrated on the full training distribution, rejects almost all partial predictions when n < 100 — precisely the regime of real-time network monitoring.

Consequence: OR/WEIGHTED → ECONOMY-K alone. AND → earliness = 0.

**Scientific value:** This is a meaningful negative result. It shows that TEASER's theoretical non-myopic stopping rule fails to generalise to low-data streaming regimes, motivating research on calibrated confidence estimators.

In [ ]:
try:
    orch1 = pd.read_csv('../results/orchestrator_results.csv')
    orch1 = orch1.drop_duplicates(subset=['dataset','method'])
    print('Experiment 1 — TEASER + ECONOMY-K orchestration:')
    print(orch1.groupby('method')[['accuracy','earliness','hm_score']].mean().round(3).to_string())
    print('\nConclusion: OR/WEIGHTED = ECONOMY-K alone. AND = earliness 0.')
except FileNotFoundError:
    print('Run src/orchestrator.py first.')

### Experiment 2 — Two-Threshold Orchestration

Replacing TEASER with a second ECONOMY-K agent at a different threshold:
- **Agent A (Aggressive):** τ = 0.40 → triggers early, lower confidence
- **Agent B (Conservative):** τ = 0.70 → waits for confidence, more accurate

This models agents with explicitly different **risk tolerances** — directly analogous to the multi-agent cooperation problem in the thesis.

In [ ]:
try:
    orch2 = pd.read_csv('../results/orchestrator2_results.csv')

    print('=== Two-Threshold Orchestration Results ===')
    summary2 = orch2.groupby('method')[['accuracy','earliness','hm_score']].mean().round(3)
    print(summary2.sort_values('hm_score', ascending=False).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    methods = ['EK-Aggressive','ECONOMY-K','EK-Conservative',
               'Orch2-OR','Orch2-AND','Orch2-WEIGHTED']
    colors  = ['#F97316','#DC2626','#7C3AED','#059669','#0891B2','#D97706']

    ax = axes[0]
    mean_hm = orch2.groupby('method')['hm_score'].mean()
    vals = [mean_hm.get(m, 0) for m in methods]
    bars = ax.bar(methods, vals, color=colors, alpha=0.85)
    ax.bar_label(bars, fmt='%.3f', fontsize=9, padding=2)
    ax.set_xticklabels(methods, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Mean HM Score')
    ax.set_title('HM Score: Solo vs Orchestrated', fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)

    ax = axes[1]
    for m, c in zip(methods, colors):
        grp = orch2[orch2['method']==m]
        ax.scatter(grp['earliness'], grp['accuracy'],
                   c=c, label=m, s=80, alpha=0.85)
    ax.set_xlabel('Earliness')
    ax.set_ylabel('Accuracy')
    ax.set_title('Pareto: Two-Threshold Orchestration', fontweight='bold')
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print('Run src/orchestrator2_twothreshold.py first.')

### Why Simple Orchestration Fails — A Fundamental Result

With homogeneous agents (same algorithm, different thresholds):

- **OR always collapses** to the most aggressive agent — first-to-trigger wins
- **AND collapses** toward the conservative agent with *lower* earliness
- **WEIGHTED degenerates** to OR when one agent always triggers first

This is not an implementation failure — it is a **fundamental limitation of parameter diversity without algorithmic diversity**.

> To escape this degeneracy, agents must differ in their *reasoning paradigm*, not just their *stopping criterion*. This is the foundational argument for the thesis research agenda.

---
## 9. Comprehensive View

In [ ]:
try:
    img = mpimg.imread('../results/figures/fig7_comprehensive_comparison.png')
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Fig 7 — Full Experimental Landscape', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('Run src/plot_all_results.py first.')

---
## 10. Implications for the CIFRE Thesis

**Axis 1 — Multi-model orchestration:** No static single-agent deployment is optimal across operational contexts. The trilemma provides a principled objective function for adaptive agent selection.

**Axis 2 — Adaptive stopping under time constraints:** The threshold sensitivity analysis shows the earliness–reliability trade-off can be steered by a single parameter. In a multi-agent setting, this parameter could be negotiated between agents based on confidence and operational urgency.

**Axis 3 — Self-organisation under constraints:** Dataset-dependence of optimal thresholds motivates self-adaptive systems that learn stopping criteria from stream statistics, without explicit per-deployment tuning.

**Axis 4 — Genuine algorithmic diversity:** The orchestration experiments show that parameter diversity alone is insufficient. Agents differing in reasoning paradigm — causal vs correlational, symbolic vs neural, local vs global — are required for non-trivial cooperation gains.

> This benchmark is the foundation. The thesis proposes to extend it from **passive comparison** to **active orchestration** — where heterogeneous agents cooperate to collectively navigate the trilemma under real-time network constraints, mediated by semantic contracts encoded in operational knowledge graphs (NORIA-O, MITRE ATT&CK).

---
## 11. References

- Schäfer, P., & Leser, U. (2020). TEASER: Early and Accurate Time Series Classification. *Data Mining and Knowledge Discovery*, 34, 1598–1626.
- Achenchabe, Y., Bondu, A., Cornuéjols, A., & Dachraoui, A. (2021). Early Classification of Time Series: Cost-Based Optimization Criterion and Algorithms. *Machine Learning*, 110, 1481–1517.
- Mori, U., Mendiburu, A., Keogh, E., & Lozano, J. A. (2017). Reliable early classification of time series based on discriminating the classes over time. *Data Mining and Knowledge Discovery*, 31(1), 233–263.
- Dau, H. A., et al. (2018). The UCR Time Series Archive. *IEEE/CAA Journal of Automatica Sinica*.
- Tailhardat, L. et al. NORIA-O: an Ontology for Anomaly Detection and Incident Management in ICT Systems. *ESWC*, 2023.